# 🔍 Fairness Analysis: End-to-End Data Exploration

This notebook performs a comprehensive fairness analysis on a synthetic dataset, including:

1. **Group Fairness** (Demographic Parity)
2. **Disparate Impact**
3. **Individual Fairness** (k-NN Consistency)
4. **Intersectional Fairness**
5. **Visualization Dashboard**
6. **Bias Mitigation Recommendations**

## Setup & Imports

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Project root (standalone-friendly)
project_root = Path.cwd()

# Styling
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

print("✅ Setup complete")

## 1. Data Generation / Loading

Generate a synthetic dataset with **known biases** (gender and race) in the outcome variable.
This simulates real-world scenarios where historical data reflects systemic inequities.

In [ ]:
np.random.seed(42)
n_samples = 10000

df = pd.DataFrame({
    "age": np.random.randint(18, 70, n_samples),
    "gender": np.random.choice(["Male", "Female"], n_samples),
    "race": np.random.choice(["White", "Black", "Asian", "Hispanic"], n_samples),
    "income": np.random.lognormal(10.5, 0.5, n_samples),
    "credit_score": np.random.normal(700, 80, n_samples).clip(300, 850),
    "education_years": np.random.randint(8, 20, n_samples),
    "loan_amount": np.random.exponential(20000, n_samples),
})

# Create biased outcome
base_prob = 0.3
df["outcome_prob"] = base_prob
df.loc[df["gender"] == "Female", "outcome_prob"] -= 0.08
df.loc[df["race"] == "Black", "outcome_prob"] -= 0.12
df.loc[df["race"] == "Hispanic", "outcome_prob"] -= 0.06
df["outcome_prob"] += (df["income"] - df["income"].mean()) / df["income"].std() * 0.1
df["outcome_prob"] = df["outcome_prob"].clip(0.05, 0.95)
df["outcome"] = (np.random.rand(n_samples) < df["outcome_prob"]).astype(int)

SENSITIVE_ATTRS = ["gender", "race"]
TARGET = "outcome"

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Group Fairness (Demographic Parity)

Compute the **Demographic Parity Difference** for each sensitive attribute.
A DP Difference > 0.1 typically indicates significant disparity.

In [ ]:
print("⚖️ GROUP FAIRNESS")
print("=" * 50)

for attr in SENSITIVE_ATTRS:
    rates = df.groupby(attr)[TARGET].mean()
    dp_diff = rates.max() - rates.min()
    print(f"\n{attr}:")
    print(rates)
    print(f"DP Difference: {dp_diff:.4f}")
    if dp_diff > 0.1:
        print("⚠️ Significant disparity detected")

## 3. Disparate Impact

Compute the **Disparate Impact Ratio** (selection rate of unprivileged / privileged).
A ratio below 0.8 (the "four-fifths rule") is considered evidence of adverse impact.

In [ ]:
print("📊 DISPARATE IMPACT")
print("=" * 50)

for attr in SENSITIVE_ATTRS:
    rates = df.groupby(attr)[TARGET].mean()
    privileged = rates.max()
    print(f"\n{attr}:")
    for g, r in rates.items():
        ratio = r / privileged
        flag = " ⚠️ Below 4/5 rule" if ratio < 0.8 else ""
        print(f"  {g} -> {ratio:.3f}{flag}")

## 4. Individual Fairness (Consistency)

Measure **consistency**: similar individuals (in feature space) should receive similar outcomes.
We use k-nearest neighbors (k=5) and compute consistency as the fraction of neighbors
with the same outcome.

In [ ]:
print("👤 INDIVIDUAL FAIRNESS")
print("=" * 50)

features = ["age", "income", "credit_score", "education_years", "loan_amount"]
X = df[features]
y = df[TARGET]

X_scaled = StandardScaler().fit_transform(X)

sample_idx = np.random.choice(len(df), 1000, replace=False)
X_s = X_scaled[sample_idx]
y_s = y.iloc[sample_idx].values

nn = NearestNeighbors(n_neighbors=6)
nn.fit(X_s)
_, idx = nn.kneighbors(X_s)

consistency = []
for i in range(len(X_s)):
    neighbors = y_s[idx[i, 1:]]
    consistency.append(np.mean(neighbors == y_s[i]))

consistency_score = np.mean(consistency)
print(f"Consistency Score: {consistency_score:.4f}")

if consistency_score < 0.7:
    print("⚠️ Low consistency — similar individuals receive different outcomes")

## 5. Intersectional Fairness

Analyze outcomes across **intersectional groups** (gender × race) to detect
disparities hidden by single-attribute analysis.

In [ ]:
print("🔗 INTERSECTIONAL FAIRNESS")
print("=" * 50)

df["group"] = df["gender"] + "_" + df["race"]
rates = df.groupby("group")[TARGET].mean().sort_values()

print(f"\nIntersectional outcome rates:")
print(rates)
print(f"\nIntersectional disparity: {rates.max() - rates.min():.4f}")

## 6. Visualization Dashboard

A 4-panel overview of target distribution, intersectional rates, and per-attribute bias.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Target distribution
df[TARGET].value_counts().plot(kind="bar", ax=axes[0, 0], color=["steelblue", "coral"])
axes[0, 0].set_title("Target Distribution")
axes[0, 0].set_xticklabels(["Negative", "Positive"], rotation=0)

# Intersectional rates
rates.plot(kind="barh", ax=axes[0, 1], color="steelblue")
axes[0, 1].set_title("Intersectional Outcome Rates")
axes[0, 1].axvline(df[TARGET].mean(), color="red", linestyle="--", label="Overall")
axes[0, 1].legend()

# Gender bias
df.groupby("gender")[TARGET].mean().plot(kind="bar", ax=axes[1, 0], color=["coral", "steelblue"])
axes[1, 0].set_title("Gender Bias")
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)

# Race bias
df.groupby("race")[TARGET].mean().plot(kind="bar", ax=axes[1, 1], color="steelblue")
axes[1, 1].set_title("Race Bias")
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45)

plt.suptitle("Fairness Analysis Dashboard", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 7. Recommendations

Actionable suggestions based on the analysis results above.

In [ ]:
print("💡 RECOMMENDATIONS")
print("=" * 50)

if consistency_score < 0.7:
    print("• Improve individual fairness via regularization")

for attr in SENSITIVE_ATTRS:
    attr_rates = df.groupby(attr)[TARGET].mean()
    if attr_rates.max() - attr_rates.min() > 0.1:
        print(f"• Reduce bias in {attr} (reweighing / adversarial debiasing)")

print("\n🎯 Analysis complete!")